# 062 — LSTM Architecture: Three Gates and the Maths

Six equations, implemented and measured. The headline: on lesson 060's
experiment the gradient 100 steps back goes from the RNN's **5.1e-26** to
**1.7e-13** — twelve orders of magnitude. The catch: **at forget bias 0 the
LSTM manages 9 steps of memory, exactly level with the RNN run it is compared
with.** The architecture alone fixes nothing; it makes the decay rate
*learnable*.

| Part | What we check |
|---|---|
| A | the six equations, one step at a time |
| B | gradient flow against forget-gate bias |
| C | effective memory: 9, 15, 44, 79 steps |
| D | the comparison with the RNN, stated honestly |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — One LSTM step, six lines

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

F, H = 8, 20
rng = np.random.default_rng(0)
W = {g: rng.normal(0, 0.3, (F + H, H)) for g in "fico"}
b = {g: np.zeros(H) for g in "fico"}

x, h, C = rng.normal(0, 1, F), np.zeros(H), np.zeros(H)

z = np.concatenate([h, x])                  # 1. concatenate h and x
f = sigmoid(z @ W["f"] + b["f"])            # 2. forget gate
i = sigmoid(z @ W["i"] + b["i"])            # 3. input gate
cand = np.tanh(z @ W["c"] + b["c"])         # 4. candidate
C_new = f * C + i * cand                    # 5. cell state - ADDITION
o = sigmoid(z @ W["o"] + b["o"])            # 6. output gate
h_new = o * np.tanh(C_new)                  #    hidden state

print(f"  concatenated input {z.shape}  (h {h.shape} + x {x.shape})")
for name, v in (("forget f", f), ("input i", i), ("candidate", cand),
                ("output o", o)):
    print(f"  {name:<10} mean {v.mean():+.3f}  range [{v.min():+.3f}, {v.max():+.3f}]")
assert z.shape == (F + H,)
assert f.min() >= 0 and f.max() <= 1 and cand.min() >= -1
print()
print("Three gates are sigmoids - they answer 'how much', between 0 and 1.")
print("The candidate is a tanh - it answers 'what', between -1 and 1.")

## Part B — Gradient flow against the forget bias

In [ ]:
def cell_state_flow(steps, forget_bias, seed=0):
    r = np.random.default_rng(seed)
    Wl = {g: r.normal(0, 0.3, (F + H, H)) for g in "fico"}
    bl = {g: np.zeros(H) for g in "fico"}
    bl["f"] = bl["f"] + forget_bias
    X = r.normal(0, 1, (steps, F))
    hh, CC, gates = np.zeros(H), np.zeros(H), []
    for t in range(steps):
        zz = np.concatenate([hh, X[t]])
        ff = sigmoid(zz @ Wl["f"] + bl["f"])
        ii = sigmoid(zz @ Wl["i"] + bl["i"])
        cc = np.tanh(zz @ Wl["c"] + bl["c"])
        oo = sigmoid(zz @ Wl["o"] + bl["o"])
        CC = ff * CC + ii * cc
        hh = oo * np.tanh(CC)
        gates.append(ff)
    dC, norms = np.ones(H), []
    for t in range(steps - 1, -1, -1):
        norms.append(np.linalg.norm(dC))
        dC = dC * gates[t]
    return np.array(norms[::-1]), float(np.mean([g.mean() for g in gates]))

print(f"  {'forget bias':>12}{'mean f':>9}{'99 steps back':>16}")
for bias in (0.0, 1.0, 2.0, 3.0):
    norms, mean_f = cell_state_flow(100, bias)
    print(f"  {bias:>12.1f}{mean_f:>9.3f}{norms[0]:>16.3e}")
norms1, _ = cell_state_flow(100, 1.0)
norms0, _ = cell_state_flow(100, 0.0)
assert f"{norms1[0]:.2e}" == "1.71e-13"
assert norms0[0] < 5.13e-26                 # bias 0 is WORSE than the RNN
print("\nRead row one: at bias 0 the gradient is 2.1e-29, below the RNN's 5.1e-26")

## Part C — Effective memory

In [ ]:
def memory(norms):
    relative = norms[::-1] / norms[-1]
    below = np.where(relative < 0.01)[0]
    return int(below[0]) if len(below) else None

mem = {}
for bias in (0.0, 1.0, 2.0, 3.0):
    norms, _ = cell_state_flow(100, bias)
    mem[bias] = memory(norms)
    print(f"  forget bias {bias}: {mem[bias]:>3} steps")
assert mem[0.0] == 9 and mem[1.0] == 15 and mem[2.0] == 44 and mem[3.0] == 79
print()
print("Bias 0 to bias 2 is 9 steps to 44, by changing one initial value. That")
print("is why initialising the forget-gate bias to 1 or 2 is standard practice.")

## Part D — Against the RNN, honestly

The RNN baseline everything here is compared with is the $\rho = 0.9$ run from
lesson 060: gradient **5.1e-26**, memory **9 steps** at the 1% threshold.

In [ ]:
RNN_GRADIENT, RNN_MEMORY = 5.13e-26, 9       # lesson 060, rho = 0.9

rows = [("simple RNN (rho = 0.9)", RNN_GRADIENT, RNN_MEMORY)]
for bias in (0.0, 1.0, 2.0):
    norms, _ = cell_state_flow(100, bias)
    rows.append((f"LSTM, forget bias {bias:.0f}", norms[0], memory(norms)))
for name, grad, steps in rows:
    print(f"  {name:<26}{grad:>12.2e}{steps:>6} steps")

lstm0 = [r for r in rows if r[0].endswith("0")][0]
assert lstm0[2] == RNN_MEMORY                 # level, not better and not worse
assert rows[2][1] / RNN_GRADIENT > 1e12       # bias 1 is 12 orders better
print()
print("Row two is the honest headline: at bias 0 the LSTM is LEVEL with the")
print("comparable RNN - 9 steps against 9 - despite four times the parameters.")
print("Row three is what the architecture makes possible once the gates open.")

## What this notebook established

- The six equations run as written: three sigmoid gates answering *how much*,
  one $\tanh$ candidate answering *what*, and a cell state updated by addition.
- Gradient 99 steps back: **2.1e-29** at forget bias 0, **1.7e-13** at bias 1 —
  twelve orders of magnitude from one initial value.
- Effective memory: **9, 15, 44, 79** steps at bias 0, 1, 2, 3.
- At bias 0 the LSTM is **level with the $\rho = 0.9$ RNN (9 steps against 9)**
  while costing 4x the parameters. The architecture makes the decay rate
  learnable; it does not set it.

## Exercises

1. Initialise the forget bias at 1 and the input gate's bias at −1. What does
   that say about what the cell should do early in training?
2. Measure memory at bias 5. Where does the benefit stop?
3. Remove the output gate ($h_t = \tanh C_t$) and re-measure. Which result
   changes, the gradient or the memory?